## DAST

Como o DAST (Dynamic Application Security Testing, Análise Dinâmica) não lê código estático, a aplicação precisa estar implantada e funcional. Em um ecossistema Python, isso significa ter a aplicação (seja em Flask, Django ou FastAPI) rodando em um ambiente de staging ou mesmo via Docker (http://localhost:5000). O banco de dados e os serviços de rede devem estar configurados exatamente como estariam em produção.

### OWASP ZAP

OWASP ZAP é a nossa ferramenta que iremos empregar para automatizar o reconhecimento de falhas de forma dinâmica, formalizando o resultado em um relatorio.

Antes de atacar, a ferramenta precisa mapear a "superfície de ataque" da aplicação navegando automaticamente pela aplicação, clicando em links, preenchendo formulários fictícios e lendo arquivos como *robots.txt *ou documentações *Swagger*.

Isto resulta em uma árvore completa de rotas (ex: /api/usuarios, /login, /dashboard), parâmetros e métodos HTTP suportados.

Na proxima etapa  o **ZAP assume o papel do invasor automatizado**, testando vulnerabilidades conhecidas (como as do OWASP Top 10) contra os endpoints descobertos.

A ferramenta ataca os campos de entrada (parâmetros de URL, formulários POST, cabeçalhos HTTP, cookies) com payloads maliciosos.

Exemplo de injeção: Se o mapeamento encontrou uma rota `/buscar?q=livros`, o ZAP enviará dezenas de variações para este mesmo endpoint, como:

`/buscar?q=' OR '1'='1 (Tentativa de SQL Injection)`

`/buscar?q=<script>alert('xss')</script> (Tentativa de Cross-Site Scripting)`

`/buscar?q=../../../etc/passwd (Tentativa de Path Traversal)`

Como o DAST tem a desvantagem de não apontar a linha exata do código afetado, a análise de vulnerabilidades exige interpretar as evidências da comunicação HTTP.

Para confirmar se o alerta gerado pela ferramenta é uma falha real ou um falso positivo, o analista deve examinar dois artefatos no relatório:

HTTP Request (A Causa): Qual dado anômalo a ferramenta enviou para o servidor?

HTTP Response (O Efeito): Como a aplicação reagiu?

###Exemplo Prático de Resolução:

Imagine que o relatório do ZAP apontou um SQL Injection de alto risco na `rota /login.`

- Investigação da Evidência: O relatório mostra que no Request *POST*, o campo password recebeu o valor `admin' --`. No Response, a aplicação retornou um `HTTP 200 OK` e o HTML da área logada, confirmando o bypass de autenticação. Alternativamente, a resposta poderia ser um erro HTTP 500 expondo um Stack Trace do psycopg2 ou sqlite3.

- Rastreamento no Código: O desenvolvedor mapeia a rota /login de volta para a arquitetura da aplicação. Ele abre o arquivo correspondente (por exemplo, views.py no Django ou app.py no Flask) e vai até o bloco de controle dessa rota.

- Correção: Ele localiza a query SQL concatenada inseguramente e a substitui por *Prepared Statements* ou pelo uso adequado do ORM do framework.

O DAST é executado preferencialmente em pipelines CI/CD automatizados (via GitHub Actions). Se a ferramenta encontra uma vulnerabilidade crítica, ela pode "quebrar a build" (interromper o pipeline), impedindo que a versão defeituosa da aplicação seja promovida e chegue ao ambiente de produção como fizemos anteriormente.

## Exemplo

1- crie um repositorio no github com os seguintes elementeos, um `main.py` e um arquivo `dast.yml` na diretorio `.github/workflows/`

```yaml

name: Teste de Segurança DAST (OWASP ZAP)

on:
  push:
    branches: [ "main" ]

permissions:
  issues: write
  contents: read

jobs:
  analise_dast:
    runs-on: ubuntu-latest

    steps:
      # 1. Baixa o código do repositório
      - name: Checkout do Código
        uses: actions/checkout@v4
      # 2. Prepara o ambiente Python
      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      # 3. Instala dependências e sobe a aplicação em background
      # '&' é usado para rodar em background e,
      #o sleep é usado para dar tempo da aplicação subir antes do ZAP iniciar a varredura
      - name: Iniciar a Aplicação FastAPI
        run: |
          pip install fastapi uvicorn
          uvicorn main:app --host 0.0.0.0 --port 8000 &
          sleep 5

      # 1. Preparamos o arquivo(relatorio) em branco e damos permissão de escrita
      # para que o usuário 'zap' dentro do Docker consiga salvar o relatório
      - name: Preparar Permissões do Relatório
        run: |
          touch report_html.html
          chmod a+w report_html.html

      # 2. Executamos o ZAP diretamente via Docker, mapeando a rede e o volume
      - name: Varredura DAST via Docker
        run: |
          docker run -v $(pwd):/zap/wrk/:rw --network=host -t ghcr.io/zaproxy/zaproxy:stable zap-baseline.py -t http://localhost:8000 -r report_html.html || true
          # O operador '|| true' no shell impede que o pipeline quebre (Crash)
          # caso o ZAP retorne código de erro ao encontrar vulnerabilidades.

      # 3. Fazemos o upload do HTML usando a Action (v4)
      - name: Disponibilizar Relatório HTML
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: relatorio-dast-html
          path: report_html.html

```

In [ ]:
# main.py
import sqlite3
import os
from fastapi import FastAPI, Query
from fastapi.responses import HTMLResponse

app = FastAPI()

# Inicialização de um banco de dados SQLite vulnerável.
def init_db():
    conn = sqlite3.connect("universidade.db")
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS alunos (id INTEGER, nome TEXT, nota REAL, segredo TEXT)")
    cursor.execute("DELETE FROM alunos") # Limpa a base a cada restart
    cursor.execute("INSERT INTO alunos VALUES (1, 'Ana', 9.5, 'SenhaAdmin123')")
    cursor.execute("INSERT INTO alunos VALUES (2, 'Carlos', 7.0, 'SenhaUser456')")
    conn.commit()
    conn.close()

init_db()


# Rota 1: Vulnerável a Reflected XSS

@app.get("/", response_class=HTMLResponse)
def index(busca: str = Query(default="")):
    # O input do usuário (busca) é refletido diretamente no HTML sem sanitização.
    # O ZAP injetará <script>alert(1)</script> aqui e detectará o XSS.
    html_content = f"""
    <html>
        <body>
            <h1>Portal de Notas</h1>
            <p>Você buscou por: <b>{busca}</b></p>
            <form method="GET" action="/buscar">
                <input type="text" name="aluno" placeholder="Nome do aluno">
                <button type="submit">Buscar Notas</button>
            </form>
        </body>
    </html>
    """
    return HTMLResponse(content=html_content)


# Rota 2: Vulnerável a SQL Injection

@app.get("/buscar")
def buscar_aluno(aluno: str):
    conn = sqlite3.connect("universidade.db")
    cursor = conn.cursor()

    # Concatenação direta de string na query SQL.
    # O ZAP testará payloads como: ' OR '1'='1
    query = f"SELECT nome, nota FROM alunos WHERE nome = '{aluno}'"

    try:
        cursor.execute(query)
        resultados = cursor.fetchall()
        return {"resultados": resultados, "query_executada": query}
    except Exception as e:
        # Expor o erro do banco de dados (Stack Trace) facilita a vida do atacante
        return {"erro": str(e)}
    finally:
        conn.close()



Agora no github actions teremos nosso relatorio. (pode ser necessario reprocessar a ação.)

In [ ]:
### CORRIGIDO AS FALHAS APONTADAS NO RELATORIO

import sqlite3
from fastapi import FastAPI, Query, Request
from fastapi.responses import HTMLResponse

app = FastAPI()

@app.middleware("http")
async def adicionar_cabecalhos_seguranca(request: Request, call_next):
    response = await call_next(request)

    ## Proteção contra Cache (Informacional)
    response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate, private"
    response.headers["Pragma"] = "no-cache"
    response.headers["Expires"] = "0"
    # Anti-clickjacking e Anti-MIME-Sniffing
    response.headers["X-Frame-Options"] = "DENY"
    response.headers["X-Content-Type-Options"] = "nosniff"

    # Content Security Policy e Permissions Policy
    # form-action 'self' garante que formulários só enviem dados para a própria origem
    response.headers["Content-Security-Policy"] = (
        "default-src 'self'; "
        "frame-ancestors 'none'; "
        "form-action 'self'"
    )

    response.headers["Permissions-Policy"] = "geolocation=(), microphone=(), camera=()"

    # Proteções Cross-Origin
    response.headers["Cross-Origin-Embedder-Policy"] = "require-corp"
    response.headers["Cross-Origin-Opener-Policy"] = "same-origin"
    response.headers["Cross-Origin-Resource-Policy"] = "same-origin"

    return response

def init_db():
    conn = sqlite3.connect("universidade.db")
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS alunos (id INTEGER, nome TEXT, nota REAL, segredo TEXT)")
    cursor.execute("DELETE FROM alunos")
    cursor.execute("INSERT INTO alunos VALUES (1, 'Ana', 9.5, 'SenhaAdmin123')")
    conn.commit()
    conn.close()

init_db()

@app.get("/", response_class=HTMLResponse)
def index(busca: str = Query(default="")):
    html_content = f"""
    <html>
        <body>
            <h1>Portal de Notas</h1>
            <p>Você buscou por: <b>{busca}</b></p>
            <form method="GET" action="/buscar">
                <input type="text" name="aluno" placeholder="Nome do aluno">
                <button type="submit">Buscar Notas</button>
            </form>
        </body>
    </html>
    """
    return HTMLResponse(content=html_content)

@app.get("/buscar")
def buscar_aluno(aluno: str):
    conn = sqlite3.connect("universidade.db")
    cursor = conn.cursor()
    #  removemos o VAZAMENTO da query na resposta HTTP.
    query = f"SELECT nome, nota FROM alunos WHERE nome = '{aluno}'"

    try:
        cursor.execute(query)
        resultados = cursor.fetchall()
        # Removida a chave "query_executada" que expunha a query SQL ao atacante.
        return {"resultados": resultados}
    except Exception:
      # Substituído o retorno de 'str(e)' por uma mensagem genérica.
        # Nunca exponha o Stack Trace do banco de dados em produção.
        return {"erro": "Ocorreu um erro interno ao processar a busca."}
    finally:
        conn.close()

#Proximos passos



Agora parece certo, porem temos ainda o SQLi, e o ZAP não reconheceu, pois estamos no modo basico de analise, para aumentarmos a complexidade temos que atualziar para o modo `zap-full-scan.py`


```yaml
- name: Varredura DAST Completa via Docker (Active Scan)
        run: |
#          docker run -v $(pwd):/zap/wrk/:rw --network=host -t ghcr.io/zaproxy/zaproxy:stable zap-full-scan.py -t http://localhost:8000 -r report_html.html || true

```

In [ ]:


@app.get("/buscar")
def buscar_aluno(aluno: str):
    conn = sqlite3.connect("universidade.db")
    cursor = conn.cursor()

    # CORREÇÃO DO SQL INJECTION
    # 1. Removemos a f-string.
    # 2. Substituímos a variável diretamente na string por um placeholder '?'.
    # Em bancos de dados como o PostgreSQL (via psycopg2) usar-se-ia '%s',
    # mas no SQLite a convenção é a interrogação.
    query = "SELECT nome, nota FROM alunos WHERE nome = ?"

    try:
        # Passamos a variável 'aluno' como uma tupla (aluno,) no segundo argumento do método execute.
        # A separação entre instrução e dados é feita internamente pelo driver do SQLite.
        cursor.execute(query, (aluno,))

        resultados = cursor.fetchall()
        return {"resultados": resultados}
    except Exception:
        # Substituído o retorno do erro nativo por uma mensagem genérica para evitar Information Disclosure.
        return {"erro": "Ocorreu um erro interno ao processar a busca."}
    finally:
        conn.close()